# LifeJacket Agent 2 Decision Agent

This notebook runs the deterministic Agent 2 pipeline for marine mammal rescue coordination. It reads Agent 1 CSV output, enriches each incident with validation, duplicate matching, severity triage, weather context, dispatch-center selection, and alert card fields.

Core logic lives in `agent2_decision_agent.py`; this notebook is an auditable execution and review layer.

## Full Agent 2 Source Code

This cell embeds the full implementation from `agent2_decision_agent.py`, making the notebook self-contained. The command-line `main()` guard is disabled here so the notebook can run safely cell by cell.

In [ ]:
"""
LifeJacket Agent 2: Decision Agent.

This module turns structured Agent 1 incident reports into deterministic,
auditable rescue-coordination decisions. It reads a CSV, validates each report,
checks for possible duplicate incidents, enriches each row with weather context,
scores severity, recommends an operational action, and writes an enriched CSV.

The weather layer is intentionally modular. `get_weather_from_api` currently
uses a deterministic placeholder provider, but the provider interface is shaped
so OpenWeather, Tomorrow.io, or NOAA/NWS can be connected later without touching
the triage rules.
"""

from __future__ import annotations

import argparse
import json
import math
import os
import re
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Iterable
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import pandas as pd


DEFAULT_INPUT_CSV = "agent1_incident_database.csv"
DEFAULT_OUTPUT_CSV = "agent2_decision_output.csv"
DEFAULT_RESCUE_CENTER_CSV = "agent2_rescue_center_directory.csv"
DEFAULT_WEATHER_PROVIDER = os.getenv("LIFEJACKET_WEATHER_PROVIDER", "placeholder")
NWS_USER_AGENT = os.getenv("LIFEJACKET_NWS_USER_AGENT", "charlottez@uchicago.edu")

DUPLICATE_DISTANCE_KM = 1.0
DUPLICATE_TIME_HOURS = 24.0

REQUIRED_COLUMNS = [
    "incident_id",
    "photo_id",
    "animal_id",
    "report_timestamp",
    "latitude",
    "longitude",
    "place_name",
    "cv_top1_species",
    "cv_top1_confidence",
    "reported_mobility",
    "reported_breathing_status",
    "reported_visible_wound",
    "reported_bleeding",
    "reported_entanglement",
    "reported_near_people",
    "reported_near_dogs",
    "llm_likely_species",
    "llm_species_confidence",
    "llm_injury_present",
    "llm_wound_present",
    "llm_bleeding_present",
    "llm_entanglement_present",
    "llm_abnormal_posture_present",
    "llm_respiratory_distress_present",
    "llm_unresponsive_present",
    "llm_mobility_concern",
]

OUTPUT_COLUMNS = [
    "case_quality_status",
    "data_conflict_flag",
    "needs_more_info",
    "missing_info",
    "species_group",
    "case_match_status",
    "matched_incident_id",
    "match_confidence",
    "weather_temperature",
    "weather_condition",
    "weather_wind_speed",
    "weather_precipitation",
    "weather_visibility",
    "weather_alert_flag",
    "weather_risk_score",
    "weather_summary",
    "weather_provider",
    "severity_level",
    "severity_score",
    "severity_confidence",
    "triage_reason_codes",
    "triggered_rules",
    "recommended_action",
    "alert_type",
    "alert_priority",
    "volunteer_allowed",
    "coordinator_review_required",
    "dispatch_center_name",
    "dispatch_hotline",
    "dispatch_email",
    "dispatch_website",
    "dispatch_response_area",
    "dispatch_response_type",
    "dispatch_match_confidence",
    "dispatch_match_reason",
    "dispatch_alternate_centers",
    "alert_card_title",
    "alert_card_summary",
    "alert_card_key_risks",
    "alert_card_missing_info",
    "alert_card_next_steps",
]

PINNIPEDS = {
    "seal",
    "harbor seal",
    "grey seal",
    "gray seal",
    "elephant seal",
    "monk seal",
    "sea lion",
    "california sea lion",
    "stellar sea lion",
    "steller sea lion",
    "fur seal",
    "walrus",
    "pinniped",
}

CETACEANS = {
    "whale",
    "dolphin",
    "porpoise",
    "orca",
    "humpback whale",
    "gray whale",
    "grey whale",
    "minke whale",
    "pilot whale",
    "bottlenose dolphin",
    "common dolphin",
    "harbor porpoise",
    "cetacean",
}

SEA_TURTLES = {
    "sea turtle",
    "green sea turtle",
    "loggerhead",
    "loggerhead sea turtle",
    "leatherback",
    "leatherback sea turtle",
    "hawksbill",
    "olive ridley",
    "kemps ridley",
    "kemp's ridley",
    "turtle",
}

COASTAL_PLACE_TO_AREA_TERMS = {
    "santa barbara": ["santa barbara"],
    "ventura": ["ventura"],
    "malibu": ["malibu", "los angeles"],
    "topanga": ["topanga", "los angeles"],
    "los angeles": ["los angeles"],
    "san pedro": ["los angeles"],
    "laguna beach": ["orange"],
    "orange county": ["orange"],
    "san diego": ["san diego"],
    "la jolla": ["san diego"],
    "santa cruz": ["santa cruz"],
    "monterey": ["monterey"],
    "moss landing": ["monterey"],
    "san luis obispo": ["san luis obispo"],
    "morro bay": ["san luis obispo"],
    "santa maria": ["san luis obispo", "santa barbara"],
    "san francisco": ["san francisco"],
    "san mateo": ["san mateo"],
    "marin": ["marin"],
    "sonoma": ["sonoma"],
    "mendocino": ["mendocino"],
    "humboldt": ["humboldt"],
    "del norte": ["del norte"],
    "crescent city": ["del norte"],
    "arcata": ["humboldt"],
    "seattle": ["seattle", "king"],
    "west seattle": ["west seattle", "seattle", "king"],
    "vashon": ["vashon"],
    "maury": ["maury"],
    "des moines": ["des moines", "king"],
    "tacoma": ["pierce"],
    "pierce": ["pierce"],
    "kitsap": ["kitsap"],
    "whatcom": ["whatcom"],
    "san juan": ["san juan"],
    "whidbey": ["whidbey"],
    "camano": ["camano"],
    "skagit": ["skagit"],
    "snohomish": ["snohomish"],
    "port townsend": ["jefferson"],
    "port angeles": ["port angeles", "clallam"],
    "neah bay": ["makah", "neah bay"],
    "olympia": ["thurston"],
    "clatsop": ["clatsop"],
    "tillamook": ["tillamook"],
    "lincoln": ["lincoln"],
    "lane": ["lane"],
    "coos": ["coos"],
    "curry": ["curry"],
}


@dataclass(frozen=True)
class WeatherReport:
    temperature: float | None
    condition: str
    wind_speed: float | None
    precipitation: float | None
    visibility: float | None
    alert_flag: bool
    risk_score: float
    summary: str
    provider: str = "placeholder"


class WeatherProvider:
    """Interface for future weather providers."""

    def fetch(self, latitude: float, longitude: float, timestamp: datetime | None) -> WeatherReport:
        raise NotImplementedError


class PlaceholderWeatherProvider(WeatherProvider):
    """
    Deterministic local weather provider.

    It gives realistic-looking values for development and testing without
    calling a network service. Replace this class with an API-backed provider
    when credentials and provider requirements are available.
    """

    def fetch(self, latitude: float, longitude: float, timestamp: datetime | None) -> WeatherReport:
        hour = timestamp.hour if timestamp else 12
        location_factor = abs(latitude * 0.7 + longitude * 0.3)

        wind_speed = round(4.0 + (location_factor % 18.0), 1)
        precipitation = round(((location_factor + hour) % 9.0) / 10.0, 2)
        visibility = round(max(1.0, 10.0 - precipitation * 5.0 - max(0.0, wind_speed - 18.0) * 0.2), 1)
        temperature = round(48.0 + (latitude % 16.0) + (hour % 5.0), 1)

        if wind_speed >= 20.0 or visibility < 3.0 or precipitation >= 0.6:
            condition = "Poor field conditions"
        elif precipitation >= 0.3:
            condition = "Light precipitation"
        elif wind_speed >= 14.0:
            condition = "Windy"
        else:
            condition = "Fair"

        risk_score = 0.0
        if wind_speed >= 15.0:
            risk_score += 0.6
        if wind_speed >= 22.0:
            risk_score += 0.4
        if precipitation >= 0.4:
            risk_score += 0.5
        if visibility < 5.0:
            risk_score += 0.5
        risk_score = round(min(risk_score, 1.5), 2)

        alert_flag = risk_score >= 1.0
        summary = (
            f"{condition}; {temperature} F, wind {wind_speed} mph, "
            f"precipitation {precipitation} in, visibility {visibility} mi."
        )

        return WeatherReport(
            temperature=temperature,
            condition=condition,
            wind_speed=wind_speed,
            precipitation=precipitation,
            visibility=visibility,
            alert_flag=alert_flag,
            risk_score=risk_score,
            summary=summary,
            provider="placeholder",
        )


class NWSWeatherProvider(WeatherProvider):
    """
    National Weather Service provider for api.weather.gov.

    Flow from `LifeJacket_APIs.docx`:
    1. `/points/{lat},{lon}` returns metadata.
    2. The metadata `forecast` URL returns forecast periods.
    3. Active alerts are queried separately for the same point.

    NWS requires a User-Agent header. The default comes from the API notes and
    can be overridden with `LIFEJACKET_NWS_USER_AGENT`.
    """

    base_url = "https://api.weather.gov"

    def __init__(self, user_agent: str = NWS_USER_AGENT, timeout_seconds: float = 8.0) -> None:
        self.user_agent = user_agent
        self.timeout_seconds = timeout_seconds

    def fetch(self, latitude: float, longitude: float, timestamp: datetime | None) -> WeatherReport:
        try:
            point_url = f"{self.base_url}/points/{latitude:.4f},{longitude:.4f}"
            point_payload = self._fetch_json(point_url)
            forecast_url = point_payload["properties"]["forecast"]
            forecast_payload = self._fetch_json(forecast_url)
            period = self._select_forecast_period(
                forecast_payload.get("properties", {}).get("periods", []),
                timestamp,
            )
            alerts = self._fetch_alerts(latitude, longitude)
            return self._build_report(period, alerts)
        except (KeyError, IndexError, TypeError, ValueError, HTTPError, URLError, TimeoutError) as exc:
            return WeatherReport(
                temperature=None,
                condition="Weather API unavailable",
                wind_speed=None,
                precipitation=None,
                visibility=None,
                alert_flag=False,
                risk_score=0.0,
                summary=f"NWS weather unavailable; using no weather severity adjustment. Reason: {exc}",
                provider="nws",
            )

    def _fetch_json(self, url: str) -> dict[str, Any]:
        headers = {
            "Accept": "application/geo+json",
            "User-Agent": self.user_agent,
        }
        request = Request(url, headers=headers)
        with urlopen(request, timeout=self.timeout_seconds) as response:
            return json.loads(response.read().decode("utf-8"))

    def _fetch_alerts(self, latitude: float, longitude: float) -> list[dict[str, Any]]:
        query = urlencode({"point": f"{latitude:.4f},{longitude:.4f}"})
        payload = self._fetch_json(f"{self.base_url}/alerts/active?{query}")
        return payload.get("features", [])

    def _select_forecast_period(
        self,
        periods: list[dict[str, Any]],
        timestamp: datetime | None,
    ) -> dict[str, Any]:
        if not periods:
            raise ValueError("NWS forecast returned no periods")
        if timestamp is None:
            return periods[0]

        for period in periods:
            start = parse_timestamp(period.get("startTime"))
            end = parse_timestamp(period.get("endTime"))
            if start and end and start <= timestamp <= end:
                return period
        return periods[0]

    def _build_report(self, period: dict[str, Any], alerts: list[dict[str, Any]]) -> WeatherReport:
        temperature = parse_float(period.get("temperature"))
        condition = normalize_text(period.get("shortForecast")) or "Forecast available"
        wind_speed = parse_wind_speed(period.get("windSpeed"))
        wind_direction = normalize_text(period.get("windDirection"))
        precipitation = parse_probability(period.get("probabilityOfPrecipitation"))
        alert_titles = [
            normalize_text(alert.get("properties", {}).get("event"))
            for alert in alerts
            if normalize_text(alert.get("properties", {}).get("event"))
        ]

        risk_score = weather_risk_from_conditions(
            condition=condition,
            wind_speed=wind_speed,
            precipitation=precipitation,
            visibility=None,
            alert_flag=bool(alert_titles),
        )
        alert_flag = bool(alert_titles) or risk_score >= 1.0

        wind_text = "wind unavailable"
        if wind_speed is not None:
            wind_text = f"wind {wind_speed:g} mph"
            if wind_direction:
                wind_text = f"{wind_text} {wind_direction}"

        precip_text = "precipitation probability unavailable"
        if precipitation is not None:
            precip_text = f"precipitation probability {precipitation:.0%}"

        alert_text = f" Active alerts: {', '.join(alert_titles[:3])}." if alert_titles else ""
        summary = f"NWS: {condition}; {temperature} F, {wind_text}, {precip_text}.{alert_text}"

        return WeatherReport(
            temperature=temperature,
            condition=condition,
            wind_speed=wind_speed,
            precipitation=precipitation,
            visibility=None,
            alert_flag=alert_flag,
            risk_score=risk_score,
            summary=summary,
            provider="nws",
        )


def parse_wind_speed(value: Any) -> float | None:
    text = normalize_text(value)
    if not text:
        return None
    speeds = [float(match) for match in re.findall(r"\d+(?:\.\d+)?", text)]
    if not speeds:
        return None
    return round(max(speeds), 1)


def parse_probability(value: Any) -> float | None:
    if isinstance(value, dict):
        value = value.get("value")
    probability = parse_float(value)
    if probability is None:
        return None
    if probability > 1:
        probability = probability / 100.0
    return max(0.0, min(probability, 1.0))


def weather_risk_from_conditions(
    condition: str,
    wind_speed: float | None,
    precipitation: float | None,
    visibility: float | None,
    alert_flag: bool,
) -> float:
    risk_score = 0.0
    condition_text = condition.lower()

    if wind_speed is not None and wind_speed >= 15.0:
        risk_score += 0.6
    if wind_speed is not None and wind_speed >= 22.0:
        risk_score += 0.4
    if precipitation is not None and precipitation >= 0.4:
        risk_score += 0.5
    if visibility is not None and visibility < 5.0:
        risk_score += 0.5
    if any(term in condition_text for term in ("fog", "storm", "thunder", "rain", "snow", "sleet")):
        risk_score += 0.4
    if alert_flag:
        risk_score += 0.7

    return round(min(risk_score, 1.5), 2)


def build_weather_provider(provider_name: str) -> WeatherProvider:
    normalized = provider_name.strip().lower()
    if normalized in {"nws", "noaa", "weather.gov", "national_weather_service"}:
        return NWSWeatherProvider()
    if normalized in {"placeholder", "mock", "offline"}:
        return PlaceholderWeatherProvider()
    raise ValueError(f"Unsupported weather provider: {provider_name}")


def get_weather_from_api(latitude: float, longitude: float, timestamp: Any) -> WeatherReport:
    """
    Weather API wrapper used by Agent 2.

    Defaults to the deterministic placeholder provider. Set
    `LIFEJACKET_WEATHER_PROVIDER=nws` or pass `--weather-provider nws` through
    the CLI path to use the National Weather Service provider.
    """

    parsed_timestamp = parse_timestamp(timestamp)
    provider = build_weather_provider(DEFAULT_WEATHER_PROVIDER)
    return provider.fetch(latitude, longitude, parsed_timestamp)


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Run the LifeJacket Agent 2 Decision Agent.")
    parser.add_argument("--input", default=DEFAULT_INPUT_CSV, help="Agent 1 input CSV path.")
    parser.add_argument("--output", default=DEFAULT_OUTPUT_CSV, help="Agent 2 output CSV path.")
    parser.add_argument(
        "--rescue-centers",
        default=DEFAULT_RESCUE_CENTER_CSV,
        help="CSV directory of rescue centers and response areas.",
    )
    parser.add_argument(
        "--demo",
        action="store_true",
        help="Create a small demo Agent 1 CSV before running the decision pipeline.",
    )
    parser.add_argument(
        "--weather-provider",
        default=DEFAULT_WEATHER_PROVIDER,
        choices=["placeholder", "nws"],
        help="Weather provider to use. Defaults to placeholder/offline.",
    )
    return parser.parse_args()


def normalize_text(value: Any) -> str:
    if pd.isna(value):
        return ""
    return str(value).strip()


def normalize_species(value: Any) -> str:
    species = normalize_text(value).lower()
    species = species.replace("_", " ").replace("-", " ")
    return " ".join(species.split())


def normalize_lookup_text(value: Any) -> str:
    text = normalize_text(value).lower()
    text = text.replace("&", " and ")
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return " ".join(text.split())


def parse_bool(value: Any) -> bool:
    if isinstance(value, bool):
        return value
    if pd.isna(value):
        return False
    if isinstance(value, (int, float)):
        return value > 0
    text = str(value).strip().lower()
    return text in {"true", "yes", "y", "1", "present", "detected", "concern", "concerning"}


def parse_float(value: Any, default: float | None = None) -> float | None:
    if pd.isna(value) or value == "":
        return default
    try:
        return float(value)
    except (TypeError, ValueError):
        return default


def parse_timestamp(value: Any) -> datetime | None:
    if pd.isna(value) or value == "":
        return None
    parsed = pd.to_datetime(value, errors="coerce", utc=True)
    if pd.isna(parsed):
        return None
    return parsed.to_pydatetime()


def join_codes(codes: Iterable[str]) -> str:
    return ";".join(dict.fromkeys(code for code in codes if code))


def ensure_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Add absent known input columns as empty strings for stable processing."""

    for column in REQUIRED_COLUMNS:
        if column not in df.columns:
            df[column] = ""
    return df


def species_group_from_species(species: Any) -> str:
    normalized = normalize_species(species)
    if not normalized or normalized in {"unknown", "unclear", "not sure", "n/a"}:
        return "unknown"
    if normalized in PINNIPEDS or any(term in normalized for term in ("seal", "sea lion", "walrus")):
        return "pinniped"
    if normalized in CETACEANS or any(term in normalized for term in ("whale", "dolphin", "porpoise", "orca")):
        return "cetacean"
    if normalized in SEA_TURTLES or "turtle" in normalized:
        return "sea_turtle"
    return "unknown"


def choose_primary_species(row: pd.Series) -> str:
    llm_species = normalize_species(row.get("llm_likely_species"))
    cv_species = normalize_species(row.get("cv_top1_species"))
    llm_confidence = parse_float(row.get("llm_species_confidence"), 0.0) or 0.0
    cv_confidence = parse_float(row.get("cv_top1_confidence"), 0.0) or 0.0

    if llm_species and llm_species != "unknown" and llm_confidence >= 0.45:
        return llm_species
    if cv_species and cv_species != "unknown" and cv_confidence >= 0.45:
        return cv_species
    return llm_species or cv_species or "unknown"


def validate_report(row: pd.Series) -> dict[str, Any]:
    missing: list[str] = []

    if not normalize_text(row.get("incident_id")):
        missing.append("incident_id")
    if parse_timestamp(row.get("report_timestamp")) is None:
        missing.append("report_timestamp")
    if parse_float(row.get("latitude")) is None or parse_float(row.get("longitude")) is None:
        missing.append("latitude_longitude")
    if not normalize_text(row.get("place_name")):
        missing.append("place_name")

    primary_species = choose_primary_species(row)
    species_group = species_group_from_species(primary_species)
    if species_group == "unknown":
        missing.append("reliable_species")

    condition_fields = [
        "reported_mobility",
        "reported_breathing_status",
        "reported_visible_wound",
        "reported_bleeding",
        "reported_entanglement",
        "llm_injury_present",
        "llm_wound_present",
        "llm_bleeding_present",
        "llm_entanglement_present",
        "llm_mobility_concern",
    ]
    if not any(normalize_text(row.get(field)) for field in condition_fields):
        missing.append("animal_condition")

    cv_species = normalize_species(row.get("cv_top1_species"))
    llm_species = normalize_species(row.get("llm_likely_species"))
    cv_group = species_group_from_species(cv_species)
    llm_group = species_group_from_species(llm_species)
    cv_confidence = parse_float(row.get("cv_top1_confidence"), 0.0) or 0.0
    llm_confidence = parse_float(row.get("llm_species_confidence"), 0.0) or 0.0
    data_conflict_flag = (
        cv_group != "unknown"
        and llm_group != "unknown"
        and cv_group != llm_group
        and cv_confidence >= 0.55
        and llm_confidence >= 0.55
    )

    if data_conflict_flag:
        missing.append("species_conflict_review")

    needs_more_info = bool(missing) or parse_bool(row.get("llm_needs_followup"))
    if not missing and parse_bool(row.get("llm_needs_followup")):
        missing.append("llm_followup_requested")

    if not missing and not data_conflict_flag:
        status = "complete"
    elif {"latitude_longitude", "report_timestamp", "reliable_species"}.intersection(missing):
        status = "incomplete"
    else:
        status = "review_needed"

    return {
        "case_quality_status": status,
        "data_conflict_flag": data_conflict_flag,
        "needs_more_info": needs_more_info,
        "missing_info": join_codes(missing),
        "species_group": species_group,
    }


def haversine_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    earth_radius_km = 6371.0
    d_lat = math.radians(lat2 - lat1)
    d_lon = math.radians(lon2 - lon1)
    a = (
        math.sin(d_lat / 2) ** 2
        + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(d_lon / 2) ** 2
    )
    return earth_radius_km * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))


def duplicate_confidence(distance_km: float, time_hours: float, species_match: bool) -> float:
    distance_score = max(0.0, 1.0 - distance_km / DUPLICATE_DISTANCE_KM)
    time_score = max(0.0, 1.0 - time_hours / DUPLICATE_TIME_HOURS)
    species_score = 1.0 if species_match else 0.35
    return round(0.45 * distance_score + 0.35 * time_score + 0.20 * species_score, 2)


def find_duplicate_match(row: pd.Series, prior_cases: list[dict[str, Any]]) -> dict[str, Any]:
    lat = parse_float(row.get("latitude"))
    lon = parse_float(row.get("longitude"))
    timestamp = parse_timestamp(row.get("report_timestamp"))
    species_group = row.get("species_group", "unknown")

    if lat is None or lon is None or timestamp is None:
        return {"case_match_status": "not_evaluable", "matched_incident_id": "", "match_confidence": 0.0}

    best_match: dict[str, Any] | None = None
    best_confidence = 0.0

    for prior in prior_cases:
        distance_km = haversine_km(lat, lon, prior["latitude"], prior["longitude"])
        time_hours = abs((timestamp - prior["timestamp"]).total_seconds()) / 3600.0
        species_match = species_group == prior["species_group"] or "unknown" in {species_group, prior["species_group"]}

        if distance_km <= DUPLICATE_DISTANCE_KM and time_hours <= DUPLICATE_TIME_HOURS and species_match:
            confidence = duplicate_confidence(distance_km, time_hours, species_group == prior["species_group"])
            if confidence > best_confidence:
                best_confidence = confidence
                best_match = prior

    if not best_match:
        return {"case_match_status": "new_incident", "matched_incident_id": "", "match_confidence": 0.0}

    status = "probable_duplicate" if best_confidence >= 0.70 else "possible_duplicate"
    return {
        "case_match_status": status,
        "matched_incident_id": best_match["incident_id"],
        "match_confidence": best_confidence,
    }


def remember_case(row: pd.Series, prior_cases: list[dict[str, Any]]) -> None:
    lat = parse_float(row.get("latitude"))
    lon = parse_float(row.get("longitude"))
    timestamp = parse_timestamp(row.get("report_timestamp"))
    incident_id = normalize_text(row.get("incident_id"))
    if lat is None or lon is None or timestamp is None or not incident_id:
        return
    if row.get("case_match_status") == "probable_duplicate":
        return
    prior_cases.append(
        {
            "incident_id": incident_id,
            "latitude": lat,
            "longitude": lon,
            "timestamp": timestamp,
            "species_group": row.get("species_group", "unknown"),
        }
    )


def get_condition_flags(row: pd.Series) -> dict[str, bool]:
    reported_mobility = normalize_text(row.get("reported_mobility")).lower()
    breathing = normalize_text(row.get("reported_breathing_status")).lower()

    return {
        "injury": parse_bool(row.get("llm_injury_present")),
        "wound": parse_bool(row.get("llm_wound_present"))
        or parse_bool(row.get("reported_visible_wound"))
        or parse_bool(row.get("cv_possible_wound")),
        "bleeding": parse_bool(row.get("llm_bleeding_present"))
        or parse_bool(row.get("reported_bleeding"))
        or parse_bool(row.get("cv_visible_blood")),
        "entanglement": parse_bool(row.get("llm_entanglement_present"))
        or parse_bool(row.get("reported_entanglement")),
        "swelling": parse_bool(row.get("llm_swelling_present")),
        "abnormal_posture": parse_bool(row.get("llm_abnormal_posture_present")),
        "respiratory_distress": parse_bool(row.get("llm_respiratory_distress_present"))
        or breathing in {"labored", "distressed", "gasping", "irregular", "not breathing"},
        "unresponsive": parse_bool(row.get("llm_unresponsive_present")),
        "mobility_concern": parse_bool(row.get("llm_mobility_concern"))
        or reported_mobility in {"immobile", "limited", "struggling", "unable to move", "not moving"},
        "near_people": parse_bool(row.get("reported_near_people")),
        "near_dogs": parse_bool(row.get("reported_near_dogs")),
    }


def score_severity(row: pd.Series, weather: WeatherReport) -> dict[str, Any]:
    flags = get_condition_flags(row)
    species_group = row.get("species_group", "unknown")
    triggered_rules: list[str] = []

    if flags["entanglement"]:
        triggered_rules.append("HARD_ENTANGLEMENT")
    if flags["wound"] and flags["bleeding"]:
        triggered_rules.append("HARD_WOUND_WITH_BLEEDING")
    if flags["respiratory_distress"] and flags["unresponsive"]:
        triggered_rules.append("HARD_RESPIRATORY_DISTRESS_UNRESPONSIVE")
    if species_group == "cetacean" and flags["mobility_concern"]:
        triggered_rules.append("HARD_CETACEAN_MOBILITY_CONCERN")

    reason_codes: list[str] = []
    score = 0.0

    scoring_weights = {
        "injury": 1.4,
        "wound": 1.3,
        "bleeding": 1.4,
        "mobility_concern": 1.2,
        "respiratory_distress": 1.5,
        "abnormal_posture": 0.9,
        "swelling": 0.7,
        "near_people": 0.6,
        "near_dogs": 0.5,
    }

    for flag_name, weight in scoring_weights.items():
        if flags[flag_name]:
            score += weight
            reason_codes.append(flag_name.upper())

    weather_severity_adjustment = min(weather.risk_score * 0.35, 0.5)
    if weather_severity_adjustment > 0:
        score += weather_severity_adjustment
        reason_codes.append("WEATHER_RISK")

    if row.get("case_quality_status") in {"incomplete", "review_needed"}:
        reason_codes.append("DATA_QUALITY_REVIEW")

    score = round(score, 2)

    if triggered_rules:
        severity_level = 3
    elif score >= 6.5:
        severity_level = 3
    elif score >= 3.0:
        severity_level = 2
    elif score >= 1.0:
        severity_level = 1
    else:
        severity_level = 0

    # Weather adds complexity but should never be the sole reason for Level 3.
    non_weather_score = max(0.0, score - weather_severity_adjustment)
    if severity_level == 3 and not triggered_rules and non_weather_score < 6.5:
        severity_level = 2

    confidence = 0.92
    if row.get("case_quality_status") == "incomplete":
        confidence -= 0.25
    elif row.get("case_quality_status") == "review_needed":
        confidence -= 0.12
    if row.get("data_conflict_flag"):
        confidence -= 0.12
    if row.get("species_group") == "unknown":
        confidence -= 0.08
    confidence = round(max(0.45, confidence), 2)

    return {
        "severity_level": severity_level,
        "severity_score": score,
        "severity_confidence": confidence,
        "triage_reason_codes": join_codes(reason_codes),
        "triggered_rules": join_codes(triggered_rules),
    }


def recommend_action(severity_level: int) -> dict[str, Any]:
    actions = {
        0: ("Guidance only", "info", "low", True, False),
        1: ("Volunteer monitor", "monitor", "normal", True, True),
        2: ("Notify rescue center", "response", "high", False, True),
        3: ("Immediate escalation", "critical", "urgent", False, True),
    }
    recommended_action, alert_type, alert_priority, volunteer_allowed, review_required = actions[severity_level]
    return {
        "recommended_action": recommended_action,
        "alert_type": alert_type,
        "alert_priority": alert_priority,
        "volunteer_allowed": volunteer_allowed,
        "coordinator_review_required": review_required,
    }


def load_rescue_center_directory(path: str | Path) -> pd.DataFrame:
    directory_path = Path(path)
    if not directory_path.exists():
        return pd.DataFrame(
            columns=[
                "center_name",
                "hotline",
                "contact",
                "phone",
                "email",
                "website",
                "response_area",
                "response_type",
                "source",
            ]
        )
    return pd.read_csv(directory_path).fillna("")


def infer_response_need(row: pd.Series) -> str:
    flags = get_condition_flags(row)
    breathing = normalize_text(row.get("reported_breathing_status")).lower()
    if flags["unresponsive"] and breathing in {"not breathing", "dead", "none"}:
        return "dead"
    return "live"


def location_terms_for_report(row: pd.Series) -> set[str]:
    place = normalize_lookup_text(row.get("place_name"))
    terms = set()

    for fragment, mapped_terms in COASTAL_PLACE_TO_AREA_TERMS.items():
        if fragment in place:
            terms.update(mapped_terms)

    county_match = re.search(r"([a-z ]+?) county", place)
    if county_match:
        terms.add(county_match.group(1).strip())

    return terms


def score_response_area(response_area: str, location_terms: set[str]) -> tuple[float, list[str]]:
    area = normalize_lookup_text(response_area)
    if not area or not location_terms:
        return 0.0, []

    matched_terms = sorted(term for term in location_terms if len(term) > 2 and term in area)
    if matched_terms:
        return min(1.0, 0.55 + 0.15 * len(matched_terms)), [f"area:{term}" for term in matched_terms[:4]]

    return 0.0, []


def score_response_capability(
    response_type: str,
    species_group: str,
    response_need: str,
    flags: dict[str, bool],
) -> tuple[float, list[str]]:
    response = normalize_lookup_text(response_type)
    reasons: list[str] = []
    score = 0.0

    if species_group == "pinniped" and ("pinniped" in response or "marine mammal" in response):
        score += 0.35
        reasons.append("capability:pinniped")
    elif species_group == "cetacean" and ("cetacean" in response or "marine mammal" in response):
        score += 0.35
        reasons.append("capability:cetacean")
    elif species_group == "sea_turtle" and "sea turtle" in response:
        score += 0.35
        reasons.append("capability:sea_turtle")
    elif species_group == "unknown" and "marine mammal" in response:
        score += 0.20
        reasons.append("capability:general_marine_mammal")

    if response_need == "live":
        if "live" in response or "rescue" in response or "stabilization" in response or "stranding response" in response:
            score += 0.30
            reasons.append("need:live_response")
        elif "dead" in response:
            score -= 0.20
            reasons.append("mismatch:dead_only")
    elif response_need == "dead" and "dead" in response:
        score += 0.30
        reasons.append("need:dead_response")

    if flags["entanglement"] and "entanglement" in response:
        score += 0.20
        reasons.append("capability:entanglement")

    if "rehabilitation" in response and response_need == "live":
        score += 0.08
        reasons.append("capability:rehabilitation")
    if "transport" in response and response_need == "live":
        score += 0.05
        reasons.append("capability:transport")

    return max(0.0, min(score, 1.0)), reasons


def select_dispatch_center(row: pd.Series, rescue_centers: pd.DataFrame) -> dict[str, Any]:
    if rescue_centers.empty:
        return {
            "dispatch_center_name": "",
            "dispatch_hotline": "",
            "dispatch_email": "",
            "dispatch_website": "",
            "dispatch_response_area": "",
            "dispatch_response_type": "",
            "dispatch_match_confidence": 0.0,
            "dispatch_match_reason": "directory_unavailable",
            "dispatch_alternate_centers": "",
        }

    location_terms = location_terms_for_report(row)
    species_group = normalize_text(row.get("species_group")) or "unknown"
    flags = get_condition_flags(row)
    response_need = infer_response_need(row)

    scored: list[dict[str, Any]] = []
    for _, center in rescue_centers.iterrows():
        area_score, area_reasons = score_response_area(center.get("response_area", ""), location_terms)
        capability_score, capability_reasons = score_response_capability(
            center.get("response_type", ""),
            species_group,
            response_need,
            flags,
        )
        confidence = round(min(1.0, 0.58 * area_score + 0.42 * capability_score), 2)
        if confidence <= 0:
            continue
        scored.append(
            {
                "center": center,
                "confidence": confidence,
                "reason": join_codes(area_reasons + capability_reasons + [f"need:{response_need}"]),
            }
        )

    if not scored:
        return {
            "dispatch_center_name": "",
            "dispatch_hotline": "",
            "dispatch_email": "",
            "dispatch_website": "",
            "dispatch_response_area": "",
            "dispatch_response_type": "",
            "dispatch_match_confidence": 0.0,
            "dispatch_match_reason": "no_area_or_capability_match",
            "dispatch_alternate_centers": "",
        }

    scored.sort(key=lambda item: item["confidence"], reverse=True)
    best = scored[0]
    center = best["center"]
    alternates = [
        f"{item['center'].get('center_name')} ({item['confidence']:.2f})"
        for item in scored[1:4]
        if item["confidence"] >= 0.45
    ]

    return {
        "dispatch_center_name": center.get("center_name", ""),
        "dispatch_hotline": center.get("hotline", ""),
        "dispatch_email": center.get("email", ""),
        "dispatch_website": center.get("website", ""),
        "dispatch_response_area": center.get("response_area", ""),
        "dispatch_response_type": center.get("response_type", ""),
        "dispatch_match_confidence": best["confidence"],
        "dispatch_match_reason": best["reason"],
        "dispatch_alternate_centers": join_codes(alternates),
    }


def severity_label(severity_level: int) -> str:
    return {0: "Info", 1: "Monitor", 2: "Respond", 3: "Critical"}.get(severity_level, "Unknown")


def build_alert_card(row: pd.Series) -> dict[str, str]:
    species = choose_primary_species(row)
    species_display = species if species != "unknown" else "Unknown species"
    location = normalize_text(row.get("place_name")) or format_coordinates(row)
    severity = severity_label(int(row.get("severity_level", 0)))
    risks = row.get("triage_reason_codes") or "No immediate risk flags"
    missing_info = row.get("missing_info") or "None"
    weather = row.get("weather_summary") or "Weather unavailable"
    action = row.get("recommended_action", "Coordinator review")
    dispatch_center = normalize_text(row.get("dispatch_center_name"))
    dispatch_hotline = normalize_text(row.get("dispatch_hotline"))
    dispatch_text = (
        f" Dispatch: {dispatch_center}"
        + (f" ({dispatch_hotline})." if dispatch_hotline else ".")
        if dispatch_center
        else " Dispatch center not identified."
    )

    title = f"{severity}: {species_display.title()} near {location}"
    summary = (
        f"{species_display.title()} report at {location}. "
        f"Severity Level {row.get('severity_level')} ({severity}); recommended action: {action}. "
        f"Weather: {weather}{dispatch_text}"
    )

    next_steps = {
        0: "Send public guidance; log report; no dispatch unless new information arrives.",
        1: "Assign trained volunteer monitor; request updates and additional photos if safe.",
        2: "Notify rescue center; coordinator should review and prepare response resources.",
        3: "Escalate immediately to rescue coordinator; restrict volunteer response to safe observation.",
    }.get(int(row.get("severity_level", 0)), "Coordinator review required.")
    if dispatch_center and int(row.get("severity_level", 0)) >= 2:
        next_steps = f"{next_steps} Contact {dispatch_center} via {dispatch_hotline or 'directory contact'}."

    return {
        "alert_card_title": title,
        "alert_card_summary": summary,
        "alert_card_key_risks": risks,
        "alert_card_missing_info": missing_info,
        "alert_card_next_steps": next_steps,
    }


def format_coordinates(row: pd.Series) -> str:
    lat = parse_float(row.get("latitude"))
    lon = parse_float(row.get("longitude"))
    if lat is None or lon is None:
        return "unknown location"
    return f"{lat:.5f}, {lon:.5f}"


def enrich_report(
    row: pd.Series,
    prior_cases: list[dict[str, Any]],
    weather_provider: WeatherProvider | None = None,
    rescue_centers: pd.DataFrame | None = None,
) -> dict[str, Any]:
    enriched = row.to_dict()

    validation = validate_report(row)
    enriched.update(validation)

    row_for_matching = pd.Series(enriched)
    match = find_duplicate_match(row_for_matching, prior_cases)
    enriched.update(match)

    lat = parse_float(row.get("latitude"))
    lon = parse_float(row.get("longitude"))
    if lat is not None and lon is not None:
        parsed_timestamp = parse_timestamp(row.get("report_timestamp"))
        weather = (
            weather_provider.fetch(lat, lon, parsed_timestamp)
            if weather_provider
            else get_weather_from_api(lat, lon, row.get("report_timestamp"))
        )
    else:
        weather = WeatherReport(None, "Unavailable", None, None, None, False, 0.0, "Weather unavailable.")

    enriched.update(
        {
            "weather_temperature": weather.temperature,
            "weather_condition": weather.condition,
            "weather_wind_speed": weather.wind_speed,
            "weather_precipitation": weather.precipitation,
            "weather_visibility": weather.visibility,
            "weather_alert_flag": weather.alert_flag,
            "weather_risk_score": weather.risk_score,
            "weather_summary": weather.summary,
            "weather_provider": weather.provider,
        }
    )

    severity = score_severity(pd.Series(enriched), weather)
    enriched.update(severity)
    enriched.update(recommend_action(int(enriched["severity_level"])))
    if rescue_centers is not None:
        enriched.update(select_dispatch_center(pd.Series(enriched), rescue_centers))
    enriched.update(build_alert_card(pd.Series(enriched)))

    remember_case(pd.Series(enriched), prior_cases)
    return enriched


def run_decision_agent(
    input_csv: str | Path,
    output_csv: str | Path,
    weather_provider_name: str = DEFAULT_WEATHER_PROVIDER,
    rescue_center_csv: str | Path = DEFAULT_RESCUE_CENTER_CSV,
) -> pd.DataFrame:
    input_path = Path(input_csv)
    output_path = Path(output_csv)

    if not input_path.exists():
        raise FileNotFoundError(
            f"Input CSV not found: {input_path}. Provide Agent 1 output or run with --demo."
        )

    df = pd.read_csv(input_path)
    df = ensure_columns(df)
    if df.empty:
        output_df = pd.DataFrame(columns=list(df.columns) + OUTPUT_COLUMNS)
        output_df.to_csv(output_path, index=False)
        return output_df

    df["_sort_timestamp"] = pd.to_datetime(df["report_timestamp"], errors="coerce", utc=True)
    df = df.sort_values(["_sort_timestamp", "incident_id"], na_position="last").drop(columns=["_sort_timestamp"])

    prior_cases: list[dict[str, Any]] = []
    weather_provider = build_weather_provider(weather_provider_name)
    rescue_centers = load_rescue_center_directory(rescue_center_csv)
    enriched_rows = [
        enrich_report(row, prior_cases, weather_provider, rescue_centers)
        for _, row in df.iterrows()
    ]
    output_df = pd.DataFrame(enriched_rows)
    output_df.to_csv(output_path, index=False)
    return output_df


def create_demo_input(path: str | Path) -> None:
    demo_rows = [
        {
            "incident_id": "LJ-0001",
            "photo_id": "IMG-1001",
            "animal_id": "",
            "report_timestamp": "2026-05-18T14:05:00Z",
            "latitude": 36.9512,
            "longitude": -122.0263,
            "place_name": "Main Beach",
            "location_source": "gps",
            "cv_top1_species": "California sea lion",
            "cv_top1_confidence": 0.82,
            "reported_mobility": "limited",
            "reported_breathing_status": "normal",
            "reported_visible_wound": "yes",
            "reported_bleeding": "yes",
            "reported_entanglement": "no",
            "reported_near_people": "yes",
            "reported_near_dogs": "no",
            "llm_likely_species": "sea lion",
            "llm_species_confidence": 0.77,
            "llm_needs_followup": "no",
            "llm_injury_present": "yes",
            "llm_wound_present": "yes",
            "llm_bleeding_present": "yes",
            "llm_entanglement_present": "no",
            "llm_swelling_present": "no",
            "llm_abnormal_posture_present": "no",
            "llm_respiratory_distress_present": "no",
            "llm_unresponsive_present": "no",
            "llm_mobility_concern": "yes",
            "llm_injury_summary": "Visible wound with bleeding and limited movement.",
            "guidance_message": "Keep distance and do not approach.",
        },
        {
            "incident_id": "LJ-0002",
            "photo_id": "IMG-1002",
            "animal_id": "",
            "report_timestamp": "2026-05-18T14:20:00Z",
            "latitude": 36.9520,
            "longitude": -122.0257,
            "place_name": "Main Beach",
            "location_source": "gps",
            "cv_top1_species": "seal",
            "cv_top1_confidence": 0.61,
            "reported_mobility": "limited",
            "reported_breathing_status": "normal",
            "reported_visible_wound": "yes",
            "reported_bleeding": "yes",
            "reported_entanglement": "no",
            "reported_near_people": "yes",
            "reported_near_dogs": "no",
            "llm_likely_species": "sea lion",
            "llm_species_confidence": 0.72,
            "llm_needs_followup": "no",
            "llm_injury_present": "yes",
            "llm_wound_present": "yes",
            "llm_bleeding_present": "yes",
            "llm_entanglement_present": "no",
            "llm_swelling_present": "no",
            "llm_abnormal_posture_present": "no",
            "llm_respiratory_distress_present": "no",
            "llm_unresponsive_present": "no",
            "llm_mobility_concern": "yes",
            "llm_injury_summary": "Likely same animal from another angle.",
            "guidance_message": "Keep distance and report changes.",
        },
        {
            "incident_id": "LJ-0003",
            "photo_id": "IMG-1003",
            "animal_id": "",
            "report_timestamp": "2026-05-18T16:10:00Z",
            "latitude": 36.6154,
            "longitude": -121.9001,
            "place_name": "Harbor Point",
            "location_source": "gps",
            "cv_top1_species": "dolphin",
            "cv_top1_confidence": 0.74,
            "reported_mobility": "immobile",
            "reported_breathing_status": "irregular",
            "reported_visible_wound": "no",
            "reported_bleeding": "no",
            "reported_entanglement": "no",
            "reported_near_people": "no",
            "reported_near_dogs": "no",
            "llm_likely_species": "dolphin",
            "llm_species_confidence": 0.69,
            "llm_needs_followup": "yes",
            "llm_followup_question": "Is the animal still in the same location?",
            "llm_injury_present": "no",
            "llm_wound_present": "no",
            "llm_bleeding_present": "no",
            "llm_entanglement_present": "no",
            "llm_swelling_present": "no",
            "llm_abnormal_posture_present": "yes",
            "llm_respiratory_distress_present": "yes",
            "llm_unresponsive_present": "no",
            "llm_mobility_concern": "yes",
            "llm_injury_summary": "Cetacean with mobility concern and abnormal posture.",
            "guidance_message": "Do not push animal back into water.",
        },
    ]
    pd.DataFrame(demo_rows).to_csv(path, index=False)


def main() -> None:
    args = parse_args()
    if args.demo:
        create_demo_input(args.input)
    output_df = run_decision_agent(args.input, args.output, args.weather_provider, args.rescue_centers)
    print(f"Wrote {len(output_df)} enriched reports to {args.output}")


# Notebook note: CLI execution is disabled here.
# Run main() manually only when using this cell as a script.

In [ ]:
from pathlib import Path
import pandas as pd

# Agent 2 functions are defined in the full source cell above.

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_colwidth', 160)

## Configure Inputs

Use `placeholder` for deterministic offline weather. Use `nws` to call the National Weather Service provider from the API notes.

In [ ]:
INPUT_CSV = Path('/Users/charlottetheghost/Desktop/agent1_incident_database.csv')
OUTPUT_CSV = Path('agent2_decision_output.csv')
RESCUE_CENTER_CSV = Path('agent2_rescue_center_directory.csv')
WEATHER_PROVIDER = 'placeholder'  # options: 'placeholder', 'nws'

INPUT_CSV.exists(), RESCUE_CENTER_CSV.exists()

## Run Agent 2

This writes the enriched CSV and returns the same data as a pandas DataFrame.

In [ ]:
decision_df = run_decision_agent(
    input_csv=INPUT_CSV,
    output_csv=OUTPUT_CSV,
    weather_provider_name=WEATHER_PROVIDER,
    rescue_center_csv=RESCUE_CENTER_CSV,
)

print(f"Wrote {len(decision_df)} enriched reports to {OUTPUT_CSV.resolve()}")
decision_df.head()

## Triage And Dispatch Review

In [ ]:
review_columns = [
    'incident_id',
    'place_name',
    'species_group',
    'case_quality_status',
    'case_match_status',
    'matched_incident_id',
    'severity_level',
    'severity_score',
    'triggered_rules',
    'recommended_action',
    'dispatch_center_name',
    'dispatch_hotline',
    'dispatch_match_confidence',
    'weather_provider',
    'weather_risk_score',
]

decision_df[review_columns]

## Alert Cards

In [ ]:
for _, row in decision_df.iterrows():
    print('=' * 88)
    print(row['alert_card_title'])
    print('- Summary:', row['alert_card_summary'])
    print('- Key risks:', row['alert_card_key_risks'])
    print('- Missing info:', row['alert_card_missing_info'])
    print('- Next steps:', row['alert_card_next_steps'])

## Rescue Center Directory Preview

In [ ]:
rescue_centers = pd.read_csv(RESCUE_CENTER_CSV)
rescue_centers[['center_name', 'hotline', 'response_area', 'response_type']].head(12)

## Optional Gemini Alert Card Copy Layer

Agent 2 should remain the source of truth for severity, dispatch, volunteer permissions, and next steps. A Gemini layer can be added after Agent 2 to rewrite alert-card text for clarity in the future app, but its output must be validated before display.

Recommended rule: Gemini may improve wording, but it may not change operational decisions.

In [ ]:
GEMINI_ALERT_CARD_SCHEMA = {
    'alert_card_title_generated': 'string',
    'alert_card_summary_generated': 'string',
    'alert_card_key_risks_generated': 'string',
    'alert_card_next_steps_generated': 'string',
    'alert_card_generation_status': 'valid | invalid | fallback',
    'alert_card_generation_model': 'string',
    'alert_card_generation_warnings': 'string',
}

GEMINI_ALERT_CARD_SCHEMA

In [ ]:
def build_gemini_alert_prompt(row):
    """Build a constrained prompt for a Gemini copy-generation layer."""
    return f"""
You are writing an operational alert card for a marine mammal rescue coordination app.

You may improve wording for clarity, but you must not change any operational decision.

Hard constraints:
- Do not change severity level: {row['severity_level']}
- Do not change recommended action: {row['recommended_action']}
- Do not change dispatch center: {row['dispatch_center_name']}
- Do not change dispatch hotline: {row['dispatch_hotline']}
- Do not say volunteers may handle the animal unless volunteer_allowed is true.
- Include public safety guidance: keep distance, do not touch, feed, move, or pour water on the animal.
- Do not invent medical facts, diagnosis, prognosis, or rescue outcomes.
- Keep language concise and suitable for coordinators.

Structured incident facts:
incident_id: {row['incident_id']}
species_group: {row['species_group']}
likely_species: {row['llm_likely_species']}
location: {row['place_name']}
severity_level: {row['severity_level']}
severity_score: {row['severity_score']}
triage_reason_codes: {row['triage_reason_codes']}
triggered_rules: {row['triggered_rules']}
recommended_action: {row['recommended_action']}
dispatch_center_name: {row['dispatch_center_name']}
dispatch_hotline: {row['dispatch_hotline']}
weather_summary: {row['weather_summary']}
missing_info: {row['missing_info']}

Return JSON only with these keys:
- alert_card_title_generated
- alert_card_summary_generated
- alert_card_key_risks_generated
- alert_card_next_steps_generated
""".strip()

print(build_gemini_alert_prompt(decision_df.iloc[0])[:1200])

In [ ]:
def mock_gemini_alert_card(row):
    """Offline stand-in for Gemini while designing the app workflow."""
    hotline = row.get('dispatch_hotline') or 'directory contact'
    center = row.get('dispatch_center_name') or 'the assigned rescue center'
    species = row.get('llm_likely_species') or row.get('species_group') or 'animal'
    location = row.get('place_name') or 'reported location'
    return {
        'alert_card_title_generated': f"Level {row['severity_level']} {species} report near {location}",
        'alert_card_summary_generated': f"{species} reported near {location}. Agent 2 recommends: {row['recommended_action']}. Dispatch contact: {center} at {hotline}.",
        'alert_card_key_risks_generated': row.get('triage_reason_codes') or 'No immediate risk flags',
        'alert_card_next_steps_generated': f"Contact {center} at {hotline}. Keep observers at least 50 yards away; do not touch, feed, move, or pour water on the animal.",
        'alert_card_generation_model': 'mock-gemini-offline',
    }

mock_gemini_alert_card(decision_df.iloc[0])

In [ ]:
def validate_generated_alert_card(row, generated):
    """Guardrail checks before generated copy is allowed into the app UI."""
    warnings = []
    combined_text = " ".join(str(value) for value in generated.values()).lower()

    expected_center = str(row.get('dispatch_center_name') or '').lower()
    expected_hotline = str(row.get('dispatch_hotline') or '').lower()
    expected_action = str(row.get('recommended_action') or '').lower()

    if expected_center and expected_center not in combined_text:
        warnings.append('missing_dispatch_center')
    if expected_hotline and expected_hotline not in combined_text:
        warnings.append('missing_dispatch_hotline')
    if expected_action and expected_action not in combined_text:
        warnings.append('missing_recommended_action')

    required_safety_terms = ['distance', 'do not touch', 'do not']
    for term in required_safety_terms:
        if term not in combined_text:
            warnings.append(f'missing_safety_language:{term}')

    forbidden_terms = ['safe to approach', 'handle the animal', 'move the animal', 'pour water']
    for term in forbidden_terms:
        if term in combined_text and term != 'pour water':
            warnings.append(f'forbidden_claim:{term}')

    status = 'valid' if not warnings else 'invalid'
    return status, ";".join(warnings)

generated_cards = []
for _, row in decision_df.iterrows():
    generated = mock_gemini_alert_card(row)
    status, warnings = validate_generated_alert_card(row, generated)
    generated['alert_card_generation_status'] = status
    generated['alert_card_generation_warnings'] = warnings
    generated_cards.append(generated)

generated_alert_df = pd.DataFrame(generated_cards)
generated_alert_df

## App Integration Note

In the final app, store both deterministic and generated alert-card fields. The deterministic fields remain the audit trail and fallback. The generated fields can power the UI only when `alert_card_generation_status == "valid"`.